# NILM – Semaine 10 : l'EMS côté serveur (fusion des semaines 8 et 9)

**Pourquoi une fusion.** Les semaines 8 et 9 ont étudié séparément les deux briques du serveur :
- **semaine 8** : le **planning du lendemain** selon le prix (chauffe-eau, batterie, heures conseillées des machines), envoyé à la carte en un message de 32 caractères ;
- **semaine 9** : l'**IA de profils d'usage** demandée par Éric (habitudes des appareils, profils de journées, recommandations chiffrées, alertes).

Les deux notebooks restent l'étude (choix des méthodes, chiffrage). Ce qui doit tourner sur le serveur, c'est **un seul programme**, l'*Energy Management System* (EMS) : `ems_serveur.py`, dans le même dossier. Il ne garde que ce qui a été retenu :
- les **règles simples** pour le planning (elles font aussi bien que l'optimiseur, semaine 8) ;
- la **moyenne des 7 derniers jours** pour la prévision (les profils ne prévoient pas mieux, semaine 9) ;
- les **habitudes, profils et recommandations** de la semaine 9.

**Comment les deux briques se servent l'une de l'autre** : elles lisent le même historique (les messages de 15 minutes de la carte). L'historique donne la prévision de consommation du planning ; les habitudes indiquent si l'heure conseillée par le planning change quelque chose pour ce foyer, et le planning de demain donne le contenu de la notification du soir (« demain jour rouge : votre lave-linge, habituellement lancé à 13 h, coûtera 0,12 € à 22 h au lieu de 0,56 € »).

```
  carte (NILM) --maison/nilm_01/q--> [enregistreur] --> historique.csv
                                                         |          |
                        prix de demain (Tempo, spot) --> [planning]  [analyse hebdo] --> conseils (Home Assistant, téléphone)
                                                         |          |
  carte (relais, batterie) <--maison/nilm_01/plan--------+   notification du soir (planning + habitudes)
```

**Ce notebook** montre le programme en action et vérifie qu'il redonne les résultats des semaines 8 et 9 :
1. l'analyse hebdomadaire sur l'historique ;
2. le planning de demain avec les vrais prix, et la notification du soir ;
3. tout l'hiver rejoué jour par jour, exactement comme le serveur le ferait, avec le coût obtenu comparé à la semaine 8 ;
4. comment le mettre en service sur le serveur de la personne A.

## 1. Le programme et l'historique
Le module `ems_serveur.py` regroupe : les tarifs et la lecture des prix (couleurs Tempo, prix spot), la prévision, les règles de planning, le codage du message, la détection des cycles, les profils (k-means), les recommandations, la notification du soir, l'enregistreur MQTT et la publication sur le broker. Tous les réglages sont dans l'objet `Config`.

L'historique utilisé ici est celui reconstitué en semaine 9 (`profils_s9/messages_15min_nilm.csv`) : une année de messages de 15 minutes de la maison UK-DALE, avec les appareils estimés par le NILM. Comme ces données datent de 2013-2014, je les rejoue sur les prix de 2024-2025 avec un décalage de 4 018 jours (574 semaines, même jour de la semaine), comme en semaines 8 et 9. Sur le vrai serveur, l'historique est celui des messages reçus et ce décalage vaut 0.

In [1]:
from pathlib import Path
import importlib, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.append(str(Path("../../flexibilite").resolve()))   # dans le dépôt, ems_serveur.py est rangé dans flexibilite/
import ems_serveur as ems
importlib.reload(ems)

HISTORIQUE = Path("profils_s9/messages_15min_nilm.csv")
DECALAGE = 4018                       # démonstration : historique 2013-2014 rejoué sur les prix 2024-2025
OUT_DIR = Path("ems_s10"); OUT_DIR.mkdir(exist_ok=True)
cfg = ems.Config()
cfg.prix_dir = Path("prix")
hist = ems.charger_historique(HISTORIQUE)
print(f"{len(hist)} messages, du {hist.index.min()} au {hist.index.max()}")
plt.rcParams["figure.figsize"] = (11, 3.2); plt.rcParams["axes.grid"] = True

35040 messages, du 2013-03-01 00:00:00 au 2014-02-28 23:45:00


## 2. L'analyse hebdomadaire
Une fois par semaine, le serveur relance l'analyse sur tout l'historique : carte d'identité du foyer, habitudes, profils de journées, coût annuel de chaque offre (ici avec un cumulus de 5 kWh par jour sur contacteur heures creuses, comme en semaine 9) et recommandations triées par gain. Le résultat est publié en JSON sur `maison/nilm_01/conseils`, que Home Assistant peut afficher et transformer en notification.

In [2]:
res = ems.analyse(hist, "Base", cfg, decalage_jours=DECALAGE, cumulus_kwh=5.0)
print("carte d'identité :", {k: round(v, 1) for k, v in res["identite"].items()})
print("coût annuel des offres (abonnement compris) :", {k: round(v) for k, v in res["couts_offres"].items()}, "-> conseillée :", res["offre_conseillee"])
print(f"profils de journées : {len(res['profils'])} | journées anormales : {len(res['anormaux'])}\n")
for r in res["recommandations"]:
    print(f"• {r['titre']}\n  {r['texte']}")
(OUT_DIR / "conseils.json").write_text(json.dumps(res["recommandations"], ensure_ascii=False, indent=1), encoding="utf-8")
res["habitudes"].round(2)

carte d'identité : {'kWh par jour': 8.3, 'veille (W)': 158.8, 'part frigo (%)': 12.9, 'part bouilloire (%)': 5.4, 'part lave-linge (%)': 7.3, 'part lave-vaisselle (%)': 4.8}
coût annuel des offres (abonnement compris) : {'Base': 1180, 'HP/HC': 1109, 'Tempo': 1066} -> conseillée : Tempo
profils de journées : 2 | journées anormales : 2

• Passer à l'offre Tempo
  Avec votre consommation, l'offre Tempo coûterait 1066 €/an contre 1180 €/an avec votre offre actuelle (Base).
• Lancer le lave-linge en heures creuses
  Vous lancez le lave-linge 5.2 fois par semaine, le plus souvent vers 13 h (99 % en heures pleines). Avec le départ conseillé dans le planning du jour : environ 19 €/an d'économie en Tempo.
• Veille : 159 W en permanence
  Vos appareils en veille consomment environ 1391 kWh par an (278 €). Chaque tranche de 10 W en moins fait gagner 18 €/an.
• Lancer le lave-vaisselle en heures creuses
  Vous lancez le lave-vaisselle 2.4 fois par semaine, le plus souvent vers 22 h (57 % en heures

,cycles par semaine,énergie d'un cycle (kWh),heure habituelle,part en heures pleines (%),part le week-end (%)
appareil,,,,,
washing_machine,5.19,0.76,13,98.89,33.33
dishwasher,2.38,1.13,22,57.26,30.65
kettle,29.44,0.10,7,98.76,27.17


## 3. Le planning de demain et la notification du soir
Chaque après-midi, vers 13 h 30, le serveur récupère la couleur Tempo et le prix spot de demain, fait la prévision de consommation (moyenne des 7 derniers jours de l'historique), applique les règles et publie le message de 32 caractères pour la carte. Il envoie aussi au téléphone la notification du soir, qui croise ce planning avec les habitudes du foyer.

La première cellule utilise les **vrais prix de demain** (si la connexion et la publication des prix le permettent). La seconde montre le même calcul sur un jour rouge de l'hiver 2024-2025, pour voir une notification complète.

In [3]:
demain = pd.Timestamp.today().normalize() + pd.Timedelta(days=1)
e0 = 0.6 * cfg.wh_cap                    # stock estimé dans la cuve à minuit
for offre in ("Tempo", "Dynamique"):
    try:
        plan = ems.planifier(demain, offre, ems.prevision_lendemain(hist, hist.index.max()), e0, cfg)
        print(f"{offre:9s} -> {plan['message']}   (topic {cfg.topic_plan})")
        print(ems.notification_du_soir(plan, res["habitudes"], cfg), "\n")
        (OUT_DIR / f"plan_demain_{offre}.json").write_text(json.dumps({"topic": cfg.topic_plan, "message": plan["message"]}), encoding="utf-8")
    except Exception as e:
        print(f"{offre:9s} -> prix de demain indisponibles ({e})")

Tempo     -> 2210;B;000003;0000AAAAA005;22;22   (topic maison/nilm_01/plan)
Demain 2026-10-22, jour bleu
- Prix stables demain : pas de précaution particulière.
- chauffe-eau : chauffe prévue à 0 h, 1 h.
- batterie : décharge de 6 h à 16 h, pendant les heures les plus chères. 

Dynamique -> 2210;-;003000;AAAA05020000;12;12   (topic maison/nilm_01/plan)
Demain 2026-10-22
- lave-vaisselle : vous le lancez souvent vers 22 h ; à 12 h le cycle coûtera 0.24 € au lieu de 0.39 €.
- chauffe-eau : chauffe prévue à 12 h, 13 h.
- batterie : décharge de 8 h à 24 h, pendant les heures les plus chères. 



In [4]:
jour_hist = pd.Timestamp("2014-01-21")            # mardi 21 janvier 2014 -> mardi 21 janvier 2025, jour rouge
cache = {}
p, c = ems.prix_horaires("Tempo", jour_hist + pd.Timedelta(days=DECALAGE), cfg, cache)
plan_r = ems.planifier(jour_hist + pd.Timedelta(days=DECALAGE), "Tempo", ems.prevision_lendemain(hist, jour_hist), e0, cfg, prix=p, couleur=c)
print("message :", plan_r["message"], "\n")
print(ems.notification_du_soir(plan_r, res["habitudes"], cfg))

message : 2101;R;000003;0002AAAAA005;22;22 

Demain 2025-01-21, jour rouge
- Jour rouge : heures pleines à 0.73 €/kWh de 6 h à 22 h, évitez four, sèche-linge et radiateurs d'appoint.
- lave-linge : vous le lancez souvent vers 13 h ; à 22 h le cycle coûtera 0.12 € au lieu de 0.56 €.
- chauffe-eau : chauffe prévue à 0 h, 1 h.
- batterie : décharge de 6 h à 17 h, pendant les heures les plus chères.


## 4. L'hiver rejoué jour par jour
Pour vérifier le programme complet, je rejoue la période de test des semaines 8 et 9 (12 novembre 2013 – 28 février 2014) **exactement comme le serveur la vivrait** : chaque jour, il ne connaît que l'historique jusqu'à la veille et les prix du lendemain, calcule le planning avec le module, et le boîtier l'exécute avec la vraie consommation de la journée. L'état de la cuve et de la batterie passe d'un jour à l'autre. Je compare au contacteur heures creuses (la référence de la semaine 8) pour les offres HP/HC, Tempo et Dynamique.

Les résultats doivent être proches de ceux des règles simples de la semaine 8 (HP/HC 2,67 €/jour, Tempo 2,78 €/jour, Dynamique 3,13 €/jour). Ils ne sont pas strictement identiques : ici la consommation vient des messages de 15 minutes en heure locale (semaine 9), alors que la semaine 8 lisait directement les mesures à la minute.

In [5]:
H = ems.horaire_par_jour(hist)
jours = [d for d in H.index if d >= pd.Timestamp("2013-11-12")]
hc = np.isin(np.arange(24), cfg.hc_hours)
lignes, plans = [], {}
for offre in ("HP/HC", "Tempo", "Dynamique"):
    cache = {}; e, s = cfg.wh_cap, 2.5; e_ref = cfg.wh_cap
    cout = cout_ref = 0.0; inconfort = 0; messages = []
    for d in jours:
        try:
            p, c = ems.prix_horaires(offre, d + pd.Timedelta(days=DECALAGE), cfg, cache)
        except RuntimeError as err:
            print(offre, "arrêté :", err); break
        plan = ems.planifier(d + pd.Timedelta(days=DECALAGE), offre, ems.prevision_lendemain(hist, d), e, cfg, prix=p, couleur=c)
        r = ems.executer_jour(plan["chauffe_eau"], plan["batterie"], H.loc[d].values, e, s, cfg)
        ref = ems.executer_jour(hc, np.zeros(24, int), H.loc[d].values, e_ref, 0.0, ems.Config(batterie=False))
        cout += (r["G"] * p).sum(); cout_ref += (ref["G"] * p).sum()
        e, s, e_ref = r["e_fin"], r["s_fin"], ref["e_fin"]; inconfort += r["inconfort"]
        messages.append({"jour": str((d + pd.Timedelta(days=DECALAGE)).date()), "message": plan["message"]})
    n = len(messages)
    lignes.append({"offre": offre, "jours": n, "contacteur HC (€/jour)": cout_ref / n, "EMS (€/jour)": cout / n,
                   "économie (%)": 100 * (1 - cout / cout_ref), "heures d'inconfort": inconfort})
    plans[offre] = messages
hiver = pd.DataFrame(lignes).set_index("offre")
hiver["semaine 8, règles simples (€/jour)"] = [{"HP/HC": 2.67, "Tempo": 2.78, "Dynamique": 3.13}[o] for o in hiver.index]
pd.DataFrame(plans["Tempo"]).to_csv(OUT_DIR / "messages_hiver_tempo.csv", index=False)
hiver.round(2)

,jours,contacteur HC (€/jour),EMS (€/jour),économie (%),heures d'inconfort,"semaine 8, règles simples (€/jour)"
offre,,,,,,
HP/HC,107,2.84,2.67,6.13,0,2.67
Tempo,107,3.31,2.80,15.33,0,2.78
Dynamique,107,3.56,3.14,11.64,0,3.13


## 5. Mise en service sur le serveur
Le programme tourne sur le serveur de la personne A (VM Home Assistant, VPS, ou plus tard un NAS), avec Python, `numpy` et `pandas`, et `paho-mqtt` pour parler au broker. Trois tâches suffisent :

| Tâche | Quand | Commande |
|---|---|---|
| Enregistrer les messages de la carte | en continu (service) | `python ems_serveur.py enregistreur --historique historique.csv --broker <broker>` |
| Planning du lendemain | tous les jours à 13 h 30 | `python ems_serveur.py planning --historique historique.csv --offre Tempo --publier` |
| Analyse et conseils | tous les lundis à 8 h | `python ems_serveur.py analyse --historique historique.csv --offre-actuelle Base --publier` |

Exemple de planification avec `cron` :
```
30 13 * * *  cd /opt/ems && python3 ems_serveur.py planning --historique historique.csv --offre Tempo --publier
0 8 * * 1    cd /opt/ems && python3 ems_serveur.py analyse  --historique historique.csv --offre-actuelle Base --publier
```
Côté Home Assistant, il suffit d'un capteur MQTT sur `maison/nilm_01/conseils` (le JSON des recommandations) et d'une automatisation qui envoie la notification sur l'application Companion. Le topic `maison/nilm_01/plan` est publié avec *retain* : la carte le récupère à son prochain réveil NB-IoT.

Points à régler avec A et B : l'adresse et les identifiants du broker, le format exact du message montant (`hhmm;frigo;bouilloire;lave_linge;lave_vaisselle;total`), et l'estimation du stock de la cuve à minuit (pour l'instant fixée à 60 %, faute de capteur de température).

## 6. Résultats et conclusion
**Le programme fusionné redonne les résultats des semaines 8 et 9.**
- **Analyse hebdomadaire** : mêmes conclusions qu'en semaine 9 : l'offre Tempo est conseillée (environ 1 066 €/an contre 1 180 €/an en Base, abonnement compris, avec le cumulus simulé), puis le décalage du lave-linge (environ 19 €/an), la réduction de la veille (159 W, 18 €/an par tranche de 10 W) et le décalage du lave-vaisselle (4 €/an). Les montants diffèrent de quelques euros de la semaine 9, car le module calcule les heures de départ des cycles à l'heure près.
- **Planning** : sur l'hiver rejoué jour par jour, avec seulement l'historique jusqu'à la veille et les prix du lendemain, l'EMS obtient des coûts proches des règles simples de la semaine 8 (tableau de la section 4), sans aucune heure d'inconfort.
- **Fusion** : la notification du soir combine le planning de demain et les habitudes du foyer. Le jour rouge du 21 janvier, elle signale que le lave-linge, habituellement lancé vers 13 h, coûterait 0,56 € au lieu de 0,12 € à 22 h, que le chauffe-eau chauffera entre 0 h et 2 h, en heures creuses, et que la batterie couvrira la maison de 6 h à 17 h.

**Ce que fait l'EMS côté serveur** : il enregistre les messages de la carte, calcule chaque après-midi le planning du lendemain et l'envoie à la carte (32 caractères), et analyse chaque semaine les habitudes du foyer pour lui envoyer des conseils chiffrés. **Entrées** : messages de 15 minutes de la carte, couleur Tempo et prix spot du lendemain. **Sorties** : message `maison/nilm_01/plan` pour la carte, JSON `maison/nilm_01/conseils` et notification du soir pour Home Assistant. **Intérêt** : c'est la partie « décision » du projet, qui transforme la désagrégation faite sur la carte en économies (jusqu'à 15 % sur l'hiver avec Tempo et batterie, 138 €/an en changeant d'offre et en décalant les machines). **Limites** : tarifs et paramètres du cumulus et de la batterie fixés dans le code, stock de la cuve estimé, testé uniquement sur une maison rejouée. **Suite** : installation sur le serveur de A, test de la voie descendante avec la carte de B, puis remplacement de l'API Tempo par la TIC du compteur.